# modular-diffusion: the three-band assembly, where it loses: the band engines in pairs, and the chain read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.14.0/colab/three_band_pairs.ipynb)

The three-band notebook (`colab/three_band.ipynb`) made band engines of Stable Diffusion 1.5 for the MIDDLE and the NOISY bands, on the shapes `tiny` and `2222/nomid/3`, and served them with the first capture's CLEAN engines: the three engines together made worse pictures than the three engines alone, each beside the model, summed. This notebook asks where and why. NOTHING IS TRAINED: every engine is one already shipped, brought back from the Hub; it runs on a card with 15 GiB.

1. THE ENGINES IN PAIRS: per shape, two of the three bands on engines and the model itself in the third (noisy and middle; middle and clean; noisy and clean), each judged by its pictures on the first 500 of the three-band notebook's held-out prompts, against the model's own pictures and against the real ones (KID, Frechet, the CLIP text-to-picture score); and the MIXED assembly (the clean and middle engines of `2222/nomid/3`, the noisy engine of `tiny`) on all 1,000. The same start noise, 50 DDIM steps, guidance 7.5, every part in half precision, as before. A parity check first: this card's pictures of the model on the first 100 prompts against the earlier ones.
2. THE CHAIN READ: for each three-band assembly, for the whole-range engine served in all three bands, and for a control whose noisy part is the untrained `tiny` engine, two sampling chains from the same start noise on 64 held-out rows: the assembly's own chain (the model asked at every step on the same state) and the model's chain (the assembly asked at every step). Per band, how far the assembly's guided answers are from the model's on each chain, and their ratio: is an engine less true to the model on the states other engines hand it than on the model's own states?

The cells: (1) install at a release tag; (2) the model, the probe and judge latents (the training latents are not needed), the first capture's clean engines, the CLIP judge, a smoke run of every job kind on a tiny random model with one chain read timed, the real judges on 16 pictures, one chain read of the model against itself on 16 rows (it must read exactly 0 at every step; it sizes the chain reads); (3) the job list: the finished jobs it reads come back from the Hub by name (the cell stops before anything runs if one did not), every new job runs and its folder is shipped at once, the report is shipped, and the machine is given back only when everything finished was shipped.

RESUME: run the cells again. Finished jobs keep their markers under `OUT`; on a new machine they come back from the Hub with their files, so nothing runs twice.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[capture,hub,data,guard,judge] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.14.0"
import os, sys, time, json, gc, hashlib, shutil, statistics, threading
import numpy as np
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 14, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import diffusers, transformers, huggingface_hub
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__,
      "| huggingface_hub", huggingface_hub.__version__, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/three_band"       # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

As the three-band notebook, without what only training needs: the model at a pinned revision, both validation parquet files of COCO 2017, the PROBE (the first 256 pictures, their content hash checked; the reads use the first 64) and the JUDGE latents (the first 1,000 pictures), each written once and skipped when present; the first capture's four clean engines; the CLIP model at a pinned revision. Then every job kind of the three-band list runs for a few steps on a tiny random model with its exactness checks, and one chain read of the tiny three-band assembly, timed, whose own chain must end where that assembly's distance read ends; the real judges run on 16 pictures with the judge's own control; the smoke is shipped; the license copy and the notice are shipped; one chain read of the model against itself on 16 probe rows must read exactly 0 at every step on both chains, and its seconds size the chain reads. No training step is run or timed: a capture training step does not fit a card of this size.

In [ ]:
# C2 - the model, the data, the inputs, the checks, the size of the job list (nothing is trained on this card)
from huggingface_hub import snapshot_download, hf_hub_download
import pyarrow.parquet as pq
from geolip.modular_diffusion import hub, run
from geolip.modular_diffusion.capture import BANDS, pictures
from geolip.modular_diffusion.capture.assemble import Assembly
from geolip.modular_diffusion.capture.data import (LatentBatches, center_crop, encode_shards, parquet_pictures, read_index,
                                                   rows_in_order, synthetic_shards)
from geolip.modular_diffusion.capture.engine import load_engine
from geolip.modular_diffusion.capture.teacher import load_teacher
from geolip.modular_diffusion.capture.train import train_band_engine
from geolip.modular_diffusion.experiments import capture
TEACHER, PROBE, JUDGE, TRAIN, RUNS = f"{OUT}/sd15", f"{OUT}/probe", f"{OUT}/judge", f"{OUT}/train", f"{OUT}/runs"
FIRST, CLIP = f"{OUT}/first_capture", f"{OUT}/clip"
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/capture/three_band"
FIRST_PREFIX = "modular_diffusion/capture/first_capture"
NOT_SHIPPED = ("checkpoint.pt", "SHIPPED", "FROM_HUB")           # the resume checkpoint and two markers; the rest ships
os.makedirs(RUNS, exist_ok=True)

# the data
snapshot_download(repo_id="stable-diffusion-v1-5/stable-diffusion-v1-5", revision="451f4fe16113bff5a5d2269ed5ad43b0592e9a14",
                  allow_patterns=["model_index.json", "scheduler/scheduler_config.json", "tokenizer/*", "text_encoder/config.json",
                                  "text_encoder/model.safetensors", "unet/config.json", "unet/diffusion_pytorch_model.safetensors",
                                  "vae/config.json", "vae/diffusion_pytorch_model.safetensors"], local_dir=TEACHER)
PARQUETS = [hf_hub_download(repo_id="lmms-lab-encoder/COCO-Caption2017", repo_type="dataset", revision="3bdd5827e243cc3084ac69a1111e69c3ab9193ff",
                            filename=f"data/val-0000{i}-of-00002.parquet", local_dir=f"{OUT}/coco") for i in (0, 1)]
N_PROBE, N_JUDGE = 256, 1000

def content_hash(pictures, captions, size=512):
    h = hashlib.sha256()
    for p, c in zip(pictures, captions):
        a = ((center_crop(p, size) + 1.0) * 127.5).round().clamp(0, 255).to(torch.uint8)    # the array the encoder sees
        h.update(a.numpy().tobytes())
        h.update(c.encode("utf-8") + b"\0")
    return h.hexdigest()

def first_captions(path):
    out = []        # the first caption of every row, read without the pictures (the rule of parquet_pictures)
    for b in pq.ParquetFile(path).iter_batches(batch_size=512, columns=["answer"]):
        for a in b.column(0).to_pylist():
            out.append("" if a is None else str((a[0] if a else "") if isinstance(a, (list, tuple)) else a))
    return out

EXPECTED_CONTENT_HASH = "93d333a65f7bbdac6ced9143ff55852273c4e2a64deb7f8f641aeb1878f35c44"           # the 256 probe pictures of the band-needs run and the first capture

def slices(sources):
    # 512 pictures at a time (decoded, handed to the encoder one by one, then dropped), so memory stays flat
    for path, first_row, end in sources:
        total = pq.ParquetFile(path).metadata.num_rows if end is None else end
        for first in range(first_row, total, 512):
            pics, _, _ = parquet_pictures(path, min(512, total - first), skip=first)
            yield from pics
            del pics

JUDGE_SOURCES = [(PARQUETS[0], 0, N_JUDGE)]
TRAIN_SOURCES = [(PARQUETS[0], N_JUDGE, None), (PARQUETS[1], 0, None)]
captions_of = lambda sources: [c for path, first, end in sources for c in first_captions(path)[first:end]]
judge_captions, train_captions = captions_of(JUDGE_SOURCES), captions_of(TRAIN_SOURCES)
JUDGE_ROWS, TRAIN_ROWS = len(judge_captions), len(train_captions)
print("judge pictures:", JUDGE_ROWS, "| expected 1,000; training pictures:", TRAIN_ROWS, "| expected 4,000")
assert JUDGE_ROWS == N_JUDGE and TRAIN_ROWS == 4000, "the judge or training rows are not the expected ones"
ready = lambda folder, n: os.path.exists(f"{folder}/index.json") and sum(read_index(folder)["counts"]) == n
probe_ready, judge_ready, train_ready = ready(PROBE, N_PROBE), ready(JUDGE, JUDGE_ROWS), True   # nothing is trained here: the training rows are not encoded
if not (probe_ready and judge_ready and train_ready):
    with_vae = load_teacher(TEACHER, device="cuda", dtype=torch.float32, vae=True)
    if not probe_ready:
        pictures_, captions, meta = parquet_pictures(PARQUETS[0], N_PROBE)
        HASH = content_hash(pictures_, captions)
        print("probe content hash", HASH, "| expected", EXPECTED_CONTENT_HASH)
        assert HASH == EXPECTED_CONTENT_HASH, "the probe pictures are not the expected data"
        encode_shards(pictures_, captions, PROBE, with_vae.vae, with_vae.tokenize, shard_size=64, batch=16, size=512, device="cuda")
        json.dump({"content_hash": HASH, "pictures": [{"row": i, "id": m.get("id"), "license": m.get("license")} for i, m in enumerate(meta)]},
                  open(f"{PROBE}/pictures.json", "w", encoding="utf-8"), indent=1)
        del pictures_
    if not judge_ready:
        # encode_shards is given a generator of pictures; a stopped encoding continues at its first unfinished shard
        encode_shards(slices(JUDGE_SOURCES), judge_captions, JUDGE, with_vae.vae, with_vae.tokenize, shard_size=500, batch=16,
                      size=512, device="cuda")
    if not train_ready:
        encode_shards(slices(TRAIN_SOURCES), train_captions, TRAIN, with_vae.vae, with_vae.tokenize, shard_size=512, batch=16,
                      size=512, device="cuda")
    del with_vae
    gc.collect()
    torch.cuda.empty_cache()
assert sum(read_index(JUDGE)["counts"]) == N_JUDGE
print("probe shards:", read_index(PROBE)["counts"], "| judge shards:", read_index(JUDGE)["counts"],
      "| training rows: not encoded (nothing is trained here)")

# the job list: (tag, arguments, the jobs it waits for, kind); the folders it reads are under `runs`, the first capture's
# clean engines under `first` (inputs, not jobs of this list)
COMMON = (f"teacher={TEACHER} device=cuda threads=4 probe_data={PROBE} probe_first=0 probe_rows=64 probe_seed=1000 "
          "probe_per_band=4 chunk=16")
TRAINING = f"variant=band_engine init=inherit text=1 batch=16 lr=5e-5 autocast=bf16 read_every=250 keep_checkpoint=0 data={TRAIN} dtype=fp32"
DISTANCE = "dtype=fp16 sample_steps=50 guidance=7.5 group=16"     # the teacher in half precision; the engines stay float32
PICTURES = (f"variant=pictures dtype=fp16 part_dtype=fp16 pic_data={JUDGE} pic_first=0 sample_steps=50 guidance=7.5 "
            f"judge=inception clip={CLIP} sheet_n=16")
SHAPES = {"2222nomid3": "2222/nomid/3", "tiny": "tiny"}
FIRST_ENGINES = [f"clean_{s}_inherit_s{k}" for s in SHAPES for k in (0, 1)]
TEACHER_PARTS = "clean=teacher,middle=teacher,noisy=teacher"

def job_list(runs, first, rows=(1000, 500, 100), blocks=10, group=50):
    at = lambda t: f"{runs}/{t}"
    big, half, small = rows
    pic = f"{PICTURES} group={group}"
    cmp = f"variant=picture_compare blocks={blocks}"
    jobs = []

    def compare(name, a, b, extra=""):
        jobs.append((name, f"{cmp} a={at(a)} b={at(b)}{extra}", [a, b], "compare"))

    def against_both(pic_job):
        x = pic_job[len("pic_"):]
        compare(f"cmp_{x}_vs_teacher", pic_job, "pic_teacher_1000")
        compare(f"cmp_{x}_vs_rows", pic_job, "pic_rows_1000")

    # the main line: the teacher's own reads and pictures, the floor, the copy check
    jobs.append(("asm_teacher", f"variant=assembly parts={TEACHER_PARTS} {DISTANCE}", [], "sample"))
    jobs.append(("pic_rows_1000", f"{pic} source=rows pic_n={big}", [], "pictures"))
    jobs.append(("pic_teacher_1000", f"{pic} parts={TEACHER_PARTS} pic_n={big} noise_seed=2000", [], "pictures"))
    jobs.append(("pic_teacher_1000_noise3000", f"{pic} parts={TEACHER_PARTS} pic_n={big} noise_seed=3000", [], "pictures"))
    jobs.append(("pic_teacher_100", f"{pic} parts={TEACHER_PARTS} pic_n={small} noise_seed=2000", [], "pictures"))
    jobs.append(("pic_copy_100", f"{pic} parts=clean=born:full,middle=born:full,noisy=born:full pic_n={small} noise_seed=2000",
                 [], "pictures"))
    compare("cmp_floor", "pic_teacher_1000_noise3000", "pic_teacher_1000", " paired=0")
    compare("cmp_floor_500", "pic_teacher_1000_noise3000", "pic_teacher_1000", f" paired=0 first_n={half}")
    compare("cmp_teacher_vs_rows", "pic_teacher_1000", "pic_rows_1000")
    compare("cmp_teacher_vs_rows_500", "pic_teacher_1000", "pic_rows_1000", f" first_n={half}")
    compare("cmp_copy_100", "pic_copy_100", "pic_teacher_100")
    # per shape: the trainings, the distance reads, the pictures, the comparisons
    for s, shape in SHAPES.items():
        mid, noi = [f"middle_{s}_inherit_s{k}" for k in (0, 1)], [f"noisy_{s}_inherit_s{k}" for k in (0, 1)]
        clean = [f"{first}/clean_{s}_inherit_s{k}" for k in (0, 1)]
        for k in (0, 1):
            jobs.append((mid[k], f"{TRAINING} shape={shape} band=middle steps=3000 seed={k}", [], "train"))
            jobs.append((noi[k], f"{TRAINING} shape={shape} band=noisy steps=3000 seed={k}", [], "train"))
        three = lambda k: f"clean={clean[k]},middle={at(mid[k])},noisy={at(noi[k])}"
        for k in (0, 1):
            jobs.append((f"asm_middle_{s}_s{k}", f"variant=assembly parts=clean=teacher,middle={at(mid[k])},noisy=teacher {DISTANCE}",
                         [mid[k]], "sample"))
            jobs.append((f"asm_noisy_{s}_s{k}", f"variant=assembly parts=clean=teacher,middle=teacher,noisy={at(noi[k])} {DISTANCE}",
                         [noi[k]], "sample"))
            jobs.append((f"asm_three_{s}_s{k}", f"variant=assembly parts={three(k)} {DISTANCE}", [mid[k], noi[k]], "sample"))
        jobs.append((f"xchg_middle_{s}", f"variant=exchange parts={three(0)} band=middle other={at(mid[1])} {DISTANCE}",
                     [mid[0], noi[0], mid[1]], "exchange"))
        jobs.append((f"xchg_noisy_{s}", f"variant=exchange parts={three(0)} band=noisy other={at(noi[1])} {DISTANCE}",
                     [mid[0], noi[0], noi[1]], "exchange"))
        jobs.append((f"off_noisy_in_middle_{s}", f"variant=assembly parts=clean=teacher,middle={at(noi[0])},noisy=teacher {DISTANCE}",
                     [noi[0]], "sample"))
        jobs.append((f"off_middle_in_noisy_{s}", f"variant=assembly parts=clean=teacher,middle=teacher,noisy={at(mid[0])} {DISTANCE}",
                     [mid[0]], "sample"))
        jobs.append((f"pic_three_{s}_s0_1000", f"{pic} parts={three(0)} pic_n={big} noise_seed=2000", [mid[0], noi[0]], "pictures"))
        against_both(f"pic_three_{s}_s0_1000")
        for pic_job, parts, needs in ((f"pic_noisy_{s}_s0_500", f"clean=teacher,middle=teacher,noisy={at(noi[0])}", [noi[0]]),
                                  (f"pic_middle_{s}_s0_500", f"clean=teacher,middle={at(mid[0])},noisy=teacher", [mid[0]]),
                                  (f"pic_clean_{s}_s0_500", f"clean={clean[0]},middle=teacher,noisy=teacher", []),
                                  (f"pic_untrained_noisy_{s}_500", f"clean=teacher,middle=teacher,noisy=born:{shape}", [])):
            jobs.append((pic_job, f"{pic} parts={parts} pic_n={half} noise_seed=2000", needs, "pictures"))
            against_both(pic_job)
    # the whole-range engine (the compute twin)
    whole = "whole_tiny_inherit_9000_s0"
    jobs.append((whole, f"{TRAINING} shape=tiny band=middle rho=1.0 steps=9000 seed=0", [], "train"))
    parts = f"clean={at(whole)},middle={at(whole)},noisy={at(whole)}"
    jobs.append(("asm_whole_tiny_9000_s0", f"variant=assembly parts={parts} {DISTANCE}", [whole], "sample"))
    jobs.append(("pic_whole_tiny_9000_s0_1000", f"{pic} parts={parts} pic_n={big} noise_seed=2000", [whole], "pictures"))
    against_both("pic_whole_tiny_9000_s0_1000")
    compare("cmp_three_tiny_vs_whole_tiny", "pic_three_tiny_s0_1000", "pic_whole_tiny_9000_s0_1000")
    for t, args, needs, kind in jobs:    # every folder a job reads is a job before it
        assert all(n in [j[0] for j in jobs[:jobs.index((t, args, needs, kind))]] for n in needs), t
    return jobs

# the first capture's four clean engines (inputs): fetched from the results repository, then loaded once on the CPU
for t in FIRST_ENGINES:
    for name in ("engine.json", "engine.safetensors", "engine_avg.safetensors"):
        dst = f"{FIRST}/{t}/{name}"
        if not os.path.exists(dst):
            got = hf_hub_download(repo_id=REPO, filename=f"{FIRST_PREFIX}/{t}/{name}", local_dir=f"{OUT}/hub_first")
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copyfile(got, dst)
cpu_teacher = load_teacher(TEACHER, device="cpu", dtype=torch.float32)
for t in FIRST_ENGINES:
    e = load_engine(cpu_teacher, f"{FIRST}/{t}", average=True)
    want = SHAPES[t.split("_")[1]]
    print(f"{t}: shape {e.shape.name}, reads the prompt: {e.text}")
    assert e.shape.name == want and e.text is False, f"{t} is not the first capture's engine of the shape {want} without the prompt"
    del e
del cpu_teacher
gc.collect()

# the text-to-picture judge at a pinned revision: its settings and weights only
snapshot_download(repo_id="openai/clip-vit-large-patch14", revision="32bd64288804d66eefd0ccbe215aa642df71cc41",
                  allow_patterns=["config.json", "model.safetensors", "preprocessor_config.json"], local_dir=CLIP)

# the smoke: every job kind of the list on the tiny random model, a few steps each, with the exactness keys; the first
# capture's engines are stood in by two-step tiny engines without the prompt
DEVICE = "cuda"
SMOKE = f"{OUT}/smoke"
shutil.rmtree(SMOKE, ignore_errors=True)
synthetic_shards(f"{SMOKE}/rows", rows=8, shard_size=3, size=16, seed=4)       # the tiny model's 16 x 16 latents
synthetic_shards(f"{SMOKE}/judge", rows=20, shard_size=7, size=16, seed=5)
SMOKE_ROWS, SMOKE_BLOCKS, SMOKE_GROUP = (20, 10, 4), 2, 2      # at least 2 rows a block of the kernel distance
SMALL = (f" tiny=1 device={DEVICE} probe_data={SMOKE}/rows probe_rows=4 probe_per_band=2 chunk=2 steps=4 batch=2 read_every=2"
         f" sample_steps=4 group={SMOKE_GROUP} save_every=0 log_every=0 pic_data={SMOKE}/judge judge=fake clip=fake sheet_n=4"
         + ("" if DEVICE == "cuda" else " autocast=none"))
for t in FIRST_ENGINES:
    run.main("capture", (f"variant=band_engine band=clean text=0 init=inherit shape={SHAPES[t.split('_')[1]]} seed={t[-1]}"
                         f" {COMMON}{SMALL} steps=2").split() + [f"out={SMOKE}/first_capture/{t}"])
smoke_jobs = job_list(f"{SMOKE}/runs", f"{SMOKE}/first_capture", SMOKE_ROWS, SMOKE_BLOCKS, SMOKE_GROUP)
smoke_jobs.append(("xchg_self", f"variant=exchange parts=clean=teacher,middle={SMOKE}/runs/middle_tiny_inherit_s0,noisy=teacher "
                   f"band=middle other={SMOKE}/runs/middle_tiny_inherit_s0 {DISTANCE}", ["middle_tiny_inherit_s0"], "exchange"))
S = {}
for t, args, needs, kind in smoke_jobs:
    S[t] = run.main("capture", f"{args} {COMMON}{SMALL}".split() + [f"out={SMOKE}/runs/{t}"])
assert S["asm_teacher"]["final_latent_change"] == 0.0, "an assembly of the model's own parts changed the final latents"
x = S["xchg_self"]["exchange"]
assert x["final_latent_change"] == 0.0 and all(v["change"] == 0.0 and v["largest_change_where_silent"] == 0.0
                                               for v in x["bands"].values()), "an engine exchanged for itself changed something"
assert all(v["largest_change_where_silent"] == 0.0 for t, *_ , k in smoke_jobs if k == "exchange"
           for v in S[t]["exchange"]["bands"].values() if v["silent_steps"]), "an exchange moved a band where it is silent"
assert S["cmp_copy_100"]["features_equal"] is True, "three copies of the full shape in half precision are not the model's pictures"
assert all(len(S[t]["curve"]) == 3 for t, _, _, k in smoke_jobs if k == "train"), "a training has no read curve"
# one chain read on the tiny model, timed: the smoke's three-band assembly of the shape tiny; its own chain must end
# where the smoke's distance read of the same assembly ends (the same final latents against the model's)
t_chain = time.time()
SMOKE_PARTS = (f"clean={SMOKE}/first_capture/clean_tiny_inherit_s0,middle={SMOKE}/runs/middle_tiny_inherit_s0,"
               f"noisy={SMOKE}/runs/noisy_tiny_inherit_s0")
SMOKE_CHAIN = run.main("capture", f"variant=chain_read parts={SMOKE_PARTS} {DISTANCE} {COMMON}{SMALL}".split()
                       + [f"out={SMOKE}/runs/chain_three_tiny_s0"])
CHAIN_SMOKE_S = time.time() - t_chain
assert SMOKE_CHAIN["final_latent_change"] == S["asm_three_tiny_s0"]["final_latent_change"], \
    "the chain read's own chain does not end where the distance read's sampling of the same assembly ends"
assert all(SMOKE_CHAIN["chain"]["bands"][b]["ratio"] is not None for b in BANDS), "a band of the tiny assembly has no ratio"
SMOKE_D1 = {"final_latent_change": SMOKE_CHAIN["final_latent_change"],
            "distance_read_final_latent_change": S["asm_three_tiny_s0"]["final_latent_change"],
            "equal": SMOKE_CHAIN["final_latent_change"] == S["asm_three_tiny_s0"]["final_latent_change"]}
print(f"smoke: one chain read on the tiny model in {CHAIN_SMOKE_S:.1f} s; it ends where the distance read ends "
      f"({SMOKE_D1['final_latent_change']!r})")
print(f"smoke: {len(smoke_jobs) + 1} jobs of every kind ran on the tiny model; the exactness keys hold")

# the real judges on the real model, 16 rows of the judge latents; then the judge's own control: the same pictures
# scored against the captions shifted by one row must score lower than against their own
JUDGE_KIND, CLIP_KIND, STEPS16, REAL_EXTRA = "inception", CLIP, 50, ""
FEATURE_DIM = 2048 if JUDGE_KIND == "inception" else 64
J16_OUT = f"{OUT}/judges16"
shutil.rmtree(J16_OUT, ignore_errors=True)
J16 = run.main("capture", (f"variant=pictures teacher={TEACHER} device={DEVICE} threads=4 parts={TEACHER_PARTS} pic_data={JUDGE} "
                           f"pic_first=0 pic_n=16 group=16 noise_seed=2000 sample_steps={STEPS16} guidance=7.5 judge={JUDGE_KIND} "
                           f"clip={CLIP_KIND} dtype=fp16 part_dtype=fp16 sheet_n=16{REAL_EXTRA}").split() + [f"out={J16_OUT}"])
with np.load(f"{J16_OUT}/pictures.npz") as d:
    feats16, clip16 = d["features"], d["clip"]
assert feats16.shape == (16, FEATURE_DIM) and np.isfinite(feats16).all(), f"the judge's features: {feats16.shape}"
assert np.isfinite(clip16).all(), "a text-to-picture score is not finite"
t16 = load_teacher(TEACHER, device=DEVICE, dtype=torch.float16, vae=True)
t16.vae.float()
x16, ids16 = rows_in_order(JUDGE, 0, 16)
shape16 = (int(t16.unet.config.in_channels), int(t16.unet.config.sample_size), int(t16.unet.config.sample_size))
z16 = pictures.sample_latents(Assembly({b: t16 for b in BANDS}), t16.text_states(ids16), t16.empty_states(1), 2000, group=16,
                              steps=STEPS16, guidance=7.5, latent_shape=shape16, alphas_cumprod=t16.alphas_cumprod)
pics16 = pictures.decode(t16.vae, z16)
clip_judge = pictures.make_clip_judge(CLIP_KIND, DEVICE)
own, shifted = clip_judge.scores(pics16, ids16), clip_judge.scores(pics16, ids16.roll(1, 0))
print(f"real judges, 16 pictures of the model: features {tuple(feats16.shape)}; CLIP against their own captions "
      f"{own.mean():.3f} (the job: {clip16.mean():.3f}), against the captions shifted by one row {shifted.mean():.3f}")
assert abs(float(own.mean()) - float(clip16.mean())) < 0.05, "the cell's pictures are not the job's"
assert shifted.mean() < own.mean(), "the CLIP judge does not score pictures higher against their own captions"
ROW_S = J16["seconds_sampling"] / 16                      # seconds a row of a sampled picture job
print(f"sampling: {ROW_S:.2f} s a row (16 rows, 50 steps, guidance 7.5, half precision)")
del t16, z16, pics16, clip_judge
gc.collect()
torch.cuda.empty_cache()

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only
    # with the tiny engines' weights, so the upload of large files is checked too
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke", "three-band assembly: smoke run", ignore=NOT_SHIPPED), \
        "the smoke folder was not shipped: check the token and the repository before the long run"

# the license copy and a notice, shipped to the root of the results folder before any trained weights
LIC = hf_hub_download(repo_id="CompVis/stable-diffusion-license", repo_type="space", filename="license.txt", local_dir=f"{OUT}/license")
NOTICE = f"{OUT}/notice"
os.makedirs(NOTICE, exist_ok=True)
shutil.copyfile(LIC, f"{NOTICE}/LICENSE_STABLE_DIFFUSION_1_5.txt")       # byte for byte
lic_bytes = open(f"{NOTICE}/LICENSE_STABLE_DIFFUSION_1_5.txt", "rb").read()
print("license copy: sha256", hashlib.sha256(lic_bytes).hexdigest(), "|", len(lic_bytes), "bytes")
NOTICE_TEXT = (
    "# Notice: files made from Stable Diffusion 1.5\n\n"
    "The `engine.safetensors` and `engine_avg.safetensors` files in the folders beside this notice are band engines: reduced "
    "copies of the Stable Diffusion 1.5 U-Net (`stable-diffusion-v1-5/stable-diffusion-v1-5`, revision `451f4fe1`). Some start "
    "from that model's weights, and all were trained on its answers or in its latent space, so they are derivatives of that "
    "model.\n\n"
    "Stable Diffusion 1.5 is published under the CreativeML Open RAIL-M license. A copy of that license is in this folder "
    "(`LICENSE_STABLE_DIFFUSION_1_5.txt`); its use restrictions (its Attachment A) travel with these files as they do with the "
    "model.\n\n"
    "These are research artifacts of a measurement, not a finished model: each engine covers one stretch of the noise range and "
    "is read beside the original model (one engine of this folder, the one whose name starts with `whole_`, was trained on "
    "the whole range for comparison).\n")
with open(f"{NOTICE}/NOTICE.md", "w", encoding="utf-8", newline="\n") as fh:
    fh.write(NOTICE_TEXT)
if REPO:
    assert hub.ship(NOTICE, REPO, PREFIX, "three-band assembly: license copy and notice"), "the license copy and notice were not shipped"

SEC = {}      # no training step is timed here: nothing is trained on this card

# one chain read of the model against itself on 16 probe rows in the distance reads' setting: it must read exactly 0 at
# every step on both chains; its seconds, scaled to the 64 probe rows, size each chain read of the job list
CHAIN_ROWS16 = 16
CHAIN16_OUT = f"{OUT}/chain16"
shutil.rmtree(CHAIN16_OUT, ignore_errors=True)
CHAIN16 = run.main("capture", (f"variant=chain_read parts={TEACHER_PARTS} {DISTANCE} {COMMON} device={DEVICE} "
                               f"probe_rows={CHAIN_ROWS16} sample_steps={STEPS16}{REAL_EXTRA}").split() + [f"out={CHAIN16_OUT}"])
c16 = CHAIN16["chain"]
assert c16["own_per_step"] == [0.0] * len(c16["steps"]) and c16["teacher_per_step"] == [0.0] * len(c16["steps"]), \
    "the model read against itself on the sampler's chain is not exactly 0 at every step"
CHAIN_S = c16["seconds"] * 64 / CHAIN_ROWS16          # seconds of one chain read on the 64 probe rows
print(f"chain read: {c16['seconds']:.1f} s on {CHAIN_ROWS16} rows (two chains, 50 steps, guidance 7.5, the model in half "
      f"precision); about {CHAIN_S / 60:.1f} min for the 64 probe rows; the model against itself reads exactly 0")
del CHAIN16
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# a job's cost here: a picture job costs its rows times the seconds a row measured above (the decoding and the judges
# included as a tenth of a second a row); a chain read costs the measured chain read on 64 rows; trainings and
# distance reads are never run here (they come back from the Hub)
def cost(args, kind):
    a = dict(x.split("=", 1) for x in args.split() if "=" in x)
    if kind == "compare":
        return 15.0
    if kind == "chain":
        return 40 + CHAIN_S
    if kind != "pictures":
        return 0.0
    n = int(a["pic_n"])
    return 40 + n * (0.1 if a.get("source") == "rows" else ROW_S + 0.1)

JOBS = job_list(RUNS, FIRST)
COST = {t: cost(a, k) for t, a, _, k in JOBS}
assert len({t for t, *_ in JOBS}) == len(JOBS) and all(n in COST for _, _, ns, _ in JOBS for n in ns)
COUNTS = {k: sum(kk == k for *_, kk in JOBS) for k in ("train", "sample", "exchange", "pictures", "compare")}
assert (COUNTS["train"], COUNTS["sample"] + COUNTS["exchange"], COUNTS["pictures"], COUNTS["compare"]) == (9, 22, 16, 28), COUNTS
INSTRUMENTS = {"seconds_a_step_batch16": SEC, "seconds_a_picture_row": ROW_S, "seconds_a_chain_read_64_rows": CHAIN_S,
               "chain_read_model_against_itself": {"rows": CHAIN_ROWS16, "seconds": c16["seconds"], "steps": len(c16["steps"]),
                                                   "own_per_step_max": max(c16["own_per_step"]),
                                                   "teacher_per_step_max": max(c16["teacher_per_step"])},
               "chain_read_tiny_smoke_seconds": CHAIN_SMOKE_S, "chain_read_tiny_smoke": SMOKE_D1,
               "card": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "training_rows": TRAIN_ROWS,
               "judge_rows": JUDGE_ROWS, "real_judges_16": {"features": list(feats16.shape), "clip_own": float(own.mean()),
                                                           "clip_shifted_by_one_row": float(shifted.mean())}}
print(f"the job list: {len(JOBS)} jobs ({COUNTS['train']} trainings, {COUNTS['sample'] + COUNTS['exchange']} distance reads, "
      f"{COUNTS['pictures']} picture jobs, {COUNTS['compare']} comparisons), about {sum(COST.values()) / 60:.0f} projected "
      f"minutes on this card with one runner")

One runner works the job list in order: the four chain reads first (they are the quick reads that decide the next step), then the parity check, the pair assemblies, the mixed assembly and their comparisons. The finished jobs this measurement reads (the middle and noisy engines of both shapes, the whole-range engine, the model's and the real pictures, the three-band `2222/nomid/3` pictures) come back from the Hub first; if one did not, the cell stops before the runner starts (nothing may be trained here). Each finished job folder is shipped at once to `REPO`, with the token from the notebook's secret `HF_TOKEN`. At the end the summary, the roster, its log and the instrument numbers are shipped to `_report_pairs`. The machine is given back (`RELEASE`) only when the report and every finished job were shipped; if an upload failed, the machine is kept and the cell names the jobs.

In [ ]:
# C3 - the job list, each finished job shipped, a progress line a minute, the summary, the machine
from geolip.modular_diffusion import jobs
RELEASE = True                     # give the machine back at the end (only when every finished job and the report were shipped)
EXTRA = ""                         # more arguments for every job (empty in a real run)
PROGRESS_S = 60

# this measurement's job list in place of the tagged one: what it reads comes back from the Hub; what it runs is new
ROWS_PAIR, ROWS_MIXED, ROWS_PARITY, BLOCKS, GROUP_PICS = 500, 1000, 100, 10, 50     # its sizes (the pictures' rows)
at = lambda t: f"{RUNS}/{t}"
pic = f"{PICTURES} group={GROUP_PICS}"
cmp = f"variant=picture_compare blocks={BLOCKS}"
KEEP = (["pic_rows_1000", "pic_teacher_1000", "pic_teacher_100", "pic_three_2222nomid3_s0_1000"]
        + [f"{b}_{s}_inherit_s0" for s in SHAPES for b in ("middle", "noisy")] + ["whole_tiny_inherit_9000_s0"])
LIST = [j for j in JOBS if j[0] in KEEP]
assert sorted(j[0] for j in LIST) == sorted(KEEP), "a job this measurement reads is not in the tagged list"
LIST = [(t, a, [n for n in ns if n in KEEP], k) for t, a, ns, k in LIST]      # a kept job waits only for kept jobs

# THE CHAIN READS FIRST (they decide the next step; each costs about four distance reads, while the picture jobs take
# hours on a small card), in the distance reads' setting: the 64 probe rows, 50 steps, guidance 7.5, groups of 16, the
# model in half precision, the engines in full: each three-band assembly, the whole-range engine in all three bands,
# and the control whose noisy part is the untrained tiny engine (its middle band's ratio must be above 1 beyond twice
# its error)
CHAIN = f"variant=chain_read {DISTANCE}"
for s in SHAPES:
    mid, noi = f"middle_{s}_inherit_s0", f"noisy_{s}_inherit_s0"
    LIST.append((f"chain_three_{s}_s0", f"{CHAIN} parts=clean={FIRST}/clean_{s}_inherit_s0,middle={at(mid)},noisy={at(noi)}",
                 [mid, noi], "chain"))
whole = "whole_tiny_inherit_9000_s0"
LIST.append(("chain_whole_tiny_9000_s0", f"{CHAIN} parts=clean={at(whole)},middle={at(whole)},noisy={at(whole)}", [whole], "chain"))
LIST.append(("chain_untrained_noisy_tiny", f"{CHAIN} parts=clean=teacher,middle={at('middle_tiny_inherit_s0')},noisy=born:tiny",
             ["middle_tiny_inherit_s0"], "chain"))

def both(pic_job):
    x = pic_job[len("pic_"):]
    LIST.append((f"cmp_{x}_vs_teacher", f"{cmp} a={at(pic_job)} b={at('pic_teacher_1000')}", [pic_job, "pic_teacher_1000"], "compare"))
    LIST.append((f"cmp_{x}_vs_rows", f"{cmp} a={at(pic_job)} b={at('pic_rows_1000')}", [pic_job, "pic_rows_1000"], "compare"))

LIST.append(("pic_teacher_100_again", f"{pic} parts={TEACHER_PARTS} pic_n={ROWS_PARITY} noise_seed=2000", [], "pictures"))
LIST.append(("cmp_parity_100", f"{cmp} a={at('pic_teacher_100_again')} b={at('pic_teacher_100')}",
            ["pic_teacher_100_again", "pic_teacher_100"], "compare"))
for s in SHAPES:
    mid, noi, clean = f"middle_{s}_inherit_s0", f"noisy_{s}_inherit_s0", f"{FIRST}/clean_{s}_inherit_s0"
    for name, parts, needs_ in ((f"pic_pair_nm_{s}_s0_500", f"clean=teacher,middle={at(mid)},noisy={at(noi)}", [mid, noi]),
                                (f"pic_pair_mc_{s}_s0_500", f"clean={clean},middle={at(mid)},noisy=teacher", [mid]),
                                (f"pic_pair_nc_{s}_s0_500", f"clean={clean},middle=teacher,noisy={at(noi)}", [noi])):
        LIST.append((name, f"{pic} parts={parts} pic_n={ROWS_PAIR} noise_seed=2000", needs_, "pictures"))
        both(name)
mixed = f"clean={FIRST}/clean_2222nomid3_inherit_s0,middle={at('middle_2222nomid3_inherit_s0')},noisy={at('noisy_tiny_inherit_s0')}"
LIST.append(("pic_mixed_s0_1000", f"{pic} parts={mixed} pic_n={ROWS_MIXED} noise_seed=2000",
            ["middle_2222nomid3_inherit_s0", "noisy_tiny_inherit_s0"], "pictures"))
both("pic_mixed_s0_1000")
LIST.append(("cmp_mixed_vs_three_2222nomid3", f"{cmp} a={at('pic_mixed_s0_1000')} b={at('pic_three_2222nomid3_s0_1000')}",
            ["pic_mixed_s0_1000", "pic_three_2222nomid3_s0_1000"], "compare"))
for i, (t, a, ns, k) in enumerate(LIST):      # every folder a job reads is a job before it
    assert all(n in [j[0] for j in LIST[:i]] for n in ns), t
JOBS = LIST
COST = {t: (0.0 if t in KEEP else cost(a, k)) for t, a, _, k in JOBS}
NEW = [t for t, *_ in JOBS if t not in KEEP]
assert len(NEW) == 2 + 6 * 3 + 3 + 1 + 4, len(NEW)
print(f"this measurement: {len(NEW)} new jobs ({sum(k == 'pictures' for t, _, _, k in JOBS if t in NEW)} picture jobs, "
      f"{sum(k == 'chain' for t, _, _, k in JOBS if t in NEW)} chain reads), "
      f"about {sum(COST.values()) / 60:.0f} projected minutes on this card; {len(KEEP)} finished jobs come back from the Hub")
kind = {t: k for t, _, _, k in JOBS}
needs = {t: n for t, _, n, _ in JOBS}
tags = [t for t, *_ in JOBS]
roster = f"{OUT}/roster.txt"
cap = lambda t: max(30, int(3 * COST[t] / 60))
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(f"{t} | {cap(t)} | geolip.modular_diffusion.run | capture {a} {COMMON}{EXTRA}" + (f" | needs={','.join(n)}" if n else "")
                       for t, a, n, _ in JOBS) + "\nEND\n")
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
ENGINE_FILES = ("engine.safetensors", "engine_avg.safetensors", "engine.json", "train.json")
PICTURE_FILES = ("pictures.npz",)

def fetch_back():
    # on a new machine: finished jobs come back with their files, so nothing runs twice; a job comes back only when what
    # it reads came back too
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in tags:
        src, dst = f"{got}/{PREFIX}/{t}", f"{RUNS}/{t}"
        if mark(t, "DONE") or not os.path.exists(f"{src}/result.json"):
            continue
        if kind[t] == "train" and not all(os.path.exists(f"{src}/{f}") for f in ENGINE_FILES):
            continue                                             # its engine files are not on the Hub: it is trained again
        if kind[t] == "pictures" and not all(os.path.exists(f"{src}/{f}") for f in PICTURE_FILES):
            continue                                             # its features are not on the Hub: it is run again
        if not all(mark(n, "DONE") for n in needs[t]):
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")

if REPO:
    fetch_back()
assert all(mark(t, "DONE") for t in KEEP), ("a job this measurement reads did not come back from the Hub: "
                                            + ", ".join(t for t in KEEP if not mark(t, "DONE")))
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")
def after_job(tag, folder, ok):
    if ok and REPO:
        if hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"three-band assembly, the engines in pairs: {tag}", ignore=NOT_SHIPPED):
            open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")      # kept across sessions on this machine
        else:
            print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)

shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
    if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
        after_job(t, f"{RUNS}/{t}", True)

t0 = time.time()
left0 = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / 60:.0f} min to run; one runner")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job, wait_s=5.0,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}))
runner.start()
while runner.is_alive():
    runner.join(PROGRESS_S)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    rate = (left0 - left) / 60 / spent if left0 > left and spent > 0 else 1.0     # projected minutes done a minute
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {left / 60 / rate:.0f} min left", flush=True)

summary = capture.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)
for f in ("roster.txt", "roster.log"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{os.path.basename(f).replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report_pairs", "three-band assembly, the engines in pairs: summary, roster and checks") if REPO else None
unshipped = [t for t in tags if mark(t, "DONE") and not shipped_or_fetched(t)]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none")
if RELEASE and shipped and not unshipped:
    from google.colab import runtime
    runtime.unassign()
else:
    print("THE MACHINE IS KEPT:", "the report was not shipped" if not shipped else
          f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)" if unshipped else "RELEASE is False")